# 5_TARGETS — Predict Heat-Flow Target Grids

Loads trained models from `4_MODEL` and predicts on Antarctica and Greenland.
Outputs one NetCDF per method per grid, plus maps and colourbar figures.

**Sections**
1. Configuration & paths
2. Load grids + artefacts
3. Helper functions (NetCDF writer, histogram, cbar+hist figure)
4. Similarity prediction + NetCDF + maps
5. QRF prediction + NetCDF + maps
6. GBM prediction + NetCDF + maps
7. Output manifest

## 1 · Configuration & paths

In [1]:
import sys, json, pickle, warnings, importlib
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'legend.frameon': False,
    'figure.facecolor': 'white',
})
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize
from pathlib import Path
from scipy.interpolate import PchipInterpolator
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from lib.agrid import Grid
import config; importlib.reload(config); from config import *

# ── output dirs ──────────────────────────────────────────────────────────
out_dir  = Path('output/targets')
fig_dir  = Path('fig/targets')
log_dir  = Path('logs')
for _d in [out_dir, fig_dir, log_dir]:
    _d.mkdir(parents=True, exist_ok=True)

# ── map appearance ────────────────────────────────────────────────────────
# All map ranges and cmaps are defined in config.py:
#   hf_cmap, hf_v_min, hf_v_max       — heat flow q
#   unc_cmap, unc_v_min, unc_v_max     — uncertainty / PI width
#   std_cmap, std_v_min, std_v_max     — std / spread
# Fallback defaults if not in config:
unc_v_min  = getattr(config, 'unc_v_min',  0)
unc_v_max  = getattr(config, 'unc_v_max',  120)
std_v_min  = getattr(config, 'std_v_min',  0)
std_v_max  = getattr(config, 'std_v_max',  60)

gridlines_kwargs = {
    'draw_labels': False, 'linewidth': 0.4,
    'color': 'gray', 'alpha': 0.5, 'linestyle': '--',
}

print(f'obs_model : {len(obs_model)} features')
print(f'out_dir   : {out_dir}')
print(f'fig_dir   : {fig_dir}')

✓ config.py v4.0 | obs_model: 21 | obs_sweep: 22 | obs: 37
✓ config.py v4.0 | obs_model: 21 | obs_sweep: 22 | obs: 37
obs_model : 21 features
out_dir   : output/targets
fig_dir   : fig/targets


## 2 · Load grids & model artefacts

In [2]:
# ── reference grid (for overplotting observations on q maps) ─────────────
ref_df = pd.read_parquet(parquet_ref)
ref = Grid(
    lats=ref_df['lat'].values, lons=ref_df['lon'].values,
    name='IHFC', crs=4326, verbose=False,
    log_file=str(log_dir / 'read_ref.log'),
)
for col in ref_df.columns:
    if col not in ('lat', 'lon'):
        ref.df[col] = ref_df[col].values
print(f'ref : {len(ref.df)} points')

# ── Antarctica ────────────────────────────────────────────────────────────
ant_df  = pd.read_parquet(parquet_ant)
ny_ant  = ant_df['y'].nunique(); nx_ant = ant_df['x'].nunique()
ant = Grid(
    lats=ant_df['lat'].values, lons=ant_df['lon'].values,
    x=ant_df['x'].values,     y=ant_df['y'].values,
    name='Antarctica', crs=3031, verbose=False,
    regular_grid=(ny_ant, nx_ant),
    log_file=str(log_dir / 'read_ant.log'),
)
for col in ant_df.columns:
    if col not in ('lat', 'lon', 'x', 'y'):
        ant.df[col] = ant_df[col].values
print(f'ant : {len(ant.df)} points  reshape={ant.reshape_tuple}')

# ── Greenland ─────────────────────────────────────────────────────────────
grl_df  = pd.read_parquet(parquet_grl)
ny_grl  = grl_df['y'].nunique(); nx_grl = grl_df['x'].nunique()
grl = Grid(
    lats=grl_df['lat'].values, lons=grl_df['lon'].values,
    x=grl_df['x'].values,     y=grl_df['y'].values,
    name='Greenland', crs=3413, verbose=False,
    regular_grid=(ny_grl, nx_grl),
    log_file=str(log_dir / 'read_grl.log'),
)
for col in grl_df.columns:
    if col not in ('lat', 'lon', 'x', 'y'):
        grl.df[col] = grl_df[col].values
print(f'grl : {len(grl.df)} points  reshape={grl.reshape_tuple}')

TARGET_GRIDS = [
    ('ant', ant_df, ant, 3031, parquet_ant),
    ('grl', grl_df, grl, 3413, parquet_grl),
]

ref : 30848 points
ant : 1779556 points  reshape=(1334, 1334)
grl : 172360 points  reshape=(556, 310)


In [ ]:
# ── load model artefacts bundle from 4_MODEL ─────────────────────────────
print('Loading model artefacts...')
artefact_path = Path('output/models/model_artefacts.pkl')
with open(artefact_path, 'rb') as f:
    A = pickle.load(f)

qrf          = A['qrf']
gbm_models   = A['gbm_models']          # {0.05: model, 0.50: model, 0.95: model}
scaler       = A['scaler']
sigma_arr    = A['sigma_arr']
K            = A['K']
X_tr         = A['X_tr']
y_tr         = A['y_tr']
w_tr         = A['w_tr']
qhat_qrf     = A.get('qhat_qrf', None)
qhat_gbm     = A.get('qhat_gbm', None)
q_clip_max   = A.get('q_clip_max', q_clip_max)  # from config as fallback

# correction splines (fitted on cal slice in 4_MODEL)
sim_correction_spline = A.get('correction_spline', None)
qrf_correction_spline = A.get('qrf_correction_spline', None)
gbm_correction_spline = A.get('gbm_correction_spline', None)

def _apply_spline(vals, spline):
    if spline is None:
        return vals.copy()
    out = np.full_like(vals, np.nan, dtype=np.float32)
    ok  = np.isfinite(vals)
    out[ok] = spline(vals[ok]).astype(np.float32)
    return out

# ── histogram bins ────────────────────────────────────────────────────────
q_edges   = np.arange(q_clip_min, q_clip_max + HIST_BIN_WIDTH, HIST_BIN_WIDTH)
n_bins    = len(q_edges) - 1
q_centers = 0.5 * (q_edges[:-1] + q_edges[1:])

print(f'QRF        : {type(qrf).__name__}')
print(f'GBM        : {len(gbm_models)} quantile models')
print(f'sigma_arr  : {len(sigma_arr)} features  K={K}')
print(f'qhat_qrf   : {qhat_qrf}   qhat_gbm: {qhat_gbm}')
print(f'Hist bins  : {n_bins}  ({q_edges[0]:.3f}–{q_edges[-1]:.3f} W/m²)')
if sim_correction_spline is None:
    print('WARNING: no sim_correction_spline in artefacts — Similarity uncorrected')
if qrf_correction_spline is None:
    print('NOTE: no qrf_correction_spline — QRF predictions uncorrected')

Loading model artefacts...


NameError: name 'Q_CLIP_MIN' is not defined

## 3 · Helper functions

In [ ]:
def make_ds_2d(df_grid, data_vars: dict, attrs: dict, crs_epsg: int,
               x_col='x', y_col='y') -> xr.Dataset:
    """Build a regular 2-D xarray Dataset from flat DataFrame + data_vars."""
    xs = np.sort(df_grid[x_col].unique())
    ys = np.sort(df_grid[y_col].unique())
    nx, ny = len(xs), len(ys)
    xi = np.searchsorted(xs, df_grid[x_col].values)
    yi = np.searchsorted(ys, df_grid[y_col].values)
    coords = {x_col: xs, y_col: ys}
    dvs = {}
    for name, arr in data_vars.items():
        if arr.ndim == 1:
            grid = np.full((ny, nx), np.nan, dtype=np.float32)
            grid[yi, xi] = arr
            dvs[name] = xr.DataArray(grid, dims=[y_col, x_col])
        elif arr.ndim == 2:
            grid = np.full((ny, nx, arr.shape[1]), np.nan, dtype=np.float32)
            grid[yi, xi, :] = arr
            dvs[name] = xr.DataArray(grid, dims=[y_col, x_col, 'bin'],
                                     coords={'bin': q_centers})
    ds = xr.Dataset(dvs, coords=coords, attrs=attrs)
    ds.attrs['crs']         = f'EPSG:{crs_epsg}'
    ds.attrs['q_clip_min']  = float(Q_CLIP_MIN)
    ds.attrs['q_clip_max']  = float(q_clip_max)
    ds.attrs['CONVENTIONS'] = 'CF-1.8'
    ds.attrs['author']      = NETCDF_AUTHOR
    return ds


def hist_from_samples(samples_2d: np.ndarray) -> np.ndarray:
    """Convert (n_pts, n_samples) to (n_pts, n_bins) normalised histograms."""
    hists = np.zeros((len(samples_2d), n_bins), dtype=np.float32)
    for i, row in enumerate(samples_2d):
        h, _ = np.histogram(row, bins=q_edges)
        s = h.sum()
        hists[i] = h / s if s > 0 else h
    return hists


def save_cbar_with_hist(data_vals, cmap, vmin, vmax, unit, label,
                        save_path, bins=60):
    """Save a standalone colourbar + histogram figure (style: 2_OBSERVABLES)."""
    fig, axes = plt.subplots(
        2, 1, figsize=(CBAR_W, CBAR_H * 2.2),
        gridspec_kw={'height_ratios': [1, 2.5], 'hspace': 0.35}
    )
    # ── colourbar strip ──────────────────────────────────────────────────
    ax_cb = axes[0]
    norm  = Normalize(vmin=vmin, vmax=vmax)
    cb    = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        cax=ax_cb, orientation='horizontal'
    )
    cb.set_label(f'{label} [{unit}]', fontsize=7)
    cb.ax.tick_params(labelsize=6)
    # ── histogram ────────────────────────────────────────────────────────
    ax_h  = axes[1]
    valid = data_vals[np.isfinite(data_vals)]
    ax_h.hist(valid, bins=bins, color='#555555', alpha=0.75, edgecolor='none')
    ax_h.axvline(np.nanmedian(valid), color='#c0392b', lw=0.9,
                 label=f'med={np.nanmedian(valid):.1f}')
    ax_h.set_xlabel(f'{label} [{unit}]', fontsize=7)
    ax_h.set_ylabel('Count', fontsize=7)
    ax_h.tick_params(labelsize=6)
    ax_h.legend(fontsize=6, loc='upper right')
    fig.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight',
                pad_inches=0.02, facecolor='white')
    plt.close(fig)


def save_maps_for_var(grid_obj, df_grid, col_name, label,
                      grid_key, method_tag,
                      cmap, vmin, vmax, unit,
                      ref_grid=None, scatter_col='q',
                      gridstep=30, show=True):
    """
    Write col_name into grid_obj.df, call agrid.mpa for the map,
    save map PNG + cbar+hist PNG.

    Returns (map_path, cbar_path).
    """
    base     = str(fig_dir / f'{method_tag}_{grid_key}_{col_name}')
    map_path = base + FIG_EXT
    cb_path  = base + '_cbar' + FIG_EXT

    # inject column
    grid_obj.df[col_name] = df_grid[col_name].values if col_name in df_grid.columns \
                             else np.nan

    figsize = MAP_W_ANT if grid_key == 'ant' else MAP_W_GRL
    figsize_h = MAP_H_ANT if grid_key == 'ant' else MAP_H_GRL
    gs = {'step': gridstep}

    map_kwargs = dict(
        data=col_name, cmap=cmap, vmin=vmin, vmax=vmax,
        coastlines=True, continents=False, cbar=False,
        ext_cbar=False, save_fig=map_path,
        no_frame=False, transparent=True, show=show,
        gridlines=gs, gridlines_kwargs=gridlines_kwargs,
        figsize=(figsize, figsize_h),
        cbarsize=(CBAR_W, CBAR_H),
    )

    # overplot IHFC observations on q maps
    if ref_grid is not None:
        # agrid scatter overlay: pass as scatter_df
        map_kwargs['scatter_df']     = ref_grid.df
        map_kwargs['scatter_col']    = scatter_col
        map_kwargs['scatter_kwargs'] = {'s': 4, 'edgecolors': 'none',
                                        'cmap': cmap, 'vmin': vmin, 'vmax': vmax,
                                        'zorder': 5, 'alpha': 0.8}

    grid_obj.mpa(**map_kwargs)

    # colourbar + histogram
    valid_vals = grid_obj.df[col_name].values
    if col_name in df_grid.columns:
        valid_vals = df_grid[col_name].values
    save_cbar_with_hist(
        valid_vals * 1e3, cmap, vmin * 1e3, vmax * 1e3,
        unit, label, cb_path
    )
    return map_path, cb_path


print('Helpers ready.')

## 4 · Similarity prediction

In [ ]:
def sim_predict_grid(X_scaled):
    """Batched Similarity kernel. Returns (q_mean, q_med, q_std, n_eff, q_hist)."""
    means, meds, stds, neffs, hists = [], [], [], [], []
    sort_idx = np.argsort(y_tr)
    y_s = y_tr[sort_idx]
    for i in range(0, len(X_scaled), BATCH_SIM):
        Xb   = X_scaled[i:i + BATCH_SIM]
        diff = (Xb[:, None, :] - X_tr[None, :, :]) / sigma_arr
        S    = np.exp(-0.5 * np.nanmean(diff ** 2, axis=2))
        S_K  = S ** K
        w    = S_K * w_tr[None, :]           # spatial density weights
        w    = w / (w.sum(axis=1, keepdims=True) + 1e-30)
        q_mean = (w * y_tr).sum(axis=1)
        q_std  = np.sqrt((w * (y_tr - q_mean[:, None]) ** 2).sum(axis=1))
        w_s    = w[:, sort_idx]
        cumw   = w_s.cumsum(axis=1)
        med_idx = np.argmax(cumw >= 0.5, axis=1)
        q_med  = y_s[med_idx]
        n_eff  = 1.0 / ((w ** 2).sum(axis=1))
        h_batch = np.zeros((len(Xb), n_bins), dtype=np.float32)
        bin_idx = np.searchsorted(q_edges[1:], y_tr)
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        for j in range(len(Xb)):
            np.add.at(h_batch[j], bin_idx, w[j])
        h_batch /= (h_batch.sum(axis=1, keepdims=True) + 1e-30)
        means.append(q_mean); meds.append(q_med)
        stds.append(q_std);   neffs.append(n_eff)
        hists.append(h_batch)
    return (
        np.concatenate(means).astype(np.float32),
        np.concatenate(meds).astype(np.float32),
        np.concatenate(stds).astype(np.float32),
        np.concatenate(neffs).astype(np.float32),
        np.vstack(hists),
    )

In [ ]:
sim_manifest = []

for grid_key, df_g_raw, grid_obj, epsg, parquet_path in TARGET_GRIDS:
    print(f'\n── Similarity [{grid_key}] ──────────────────────────')
    df_g = df_g_raw.copy()

    # ── feature matrix ──────────────────────────────────────────────────
    missing_cols = [c for c in obs_model if c not in df_g.columns]
    if missing_cols:
        print(f'  WARNING: missing columns {missing_cols} — filling NaN')
        for c in missing_cols:
            df_g[c] = np.nan

    X_g_raw  = df_g[obs_model].values.astype(np.float32)
    X_g      = scaler.transform(X_g_raw) / sigma_arr

    # ── predict ─────────────────────────────────────────────────────────
    q_mean, q_med, q_std, n_eff, q_hist = sim_predict_grid(X_g)

    q_mean_c = _apply_spline(q_mean, sim_correction_spline)
    q_med_c  = _apply_spline(q_med,  sim_correction_spline)

    # ── NetCDF ───────────────────────────────────────────────────────────
    ds = make_ds_2d(df_g, {
        'Q_mean'  : q_mean_c,
        'Q_median': q_med_c,
        'Q_std'   : q_std,
        'N_eff'   : n_eff,
        'Q_hist'  : q_hist,
    }, attrs={
        'method': 'Similarity kernel (distribution-corrected)',
        'K'     : float(K),
        'source': str(parquet_path),
    }, crs_epsg=epsg)

    nc_path = out_dir / f'sim_{grid_key}.nc'
    ds.to_netcdf(nc_path)
    print(f'  saved → {nc_path}')
    print(f'  Q_mean range : {float(q_mean_c.min())*1e3:.1f}–{float(q_mean_c.max())*1e3:.1f} mW/m²')
    print(f'  N_eff median : {float(np.nanmedian(n_eff)):.1f}')

    # ── inject prediction cols for mapping ──────────────────────────────
    df_g['Q_mean']   = q_mean_c
    df_g['Q_median'] = q_med_c
    df_g['Q_std']    = q_std
    df_g['N_eff']    = n_eff

    gs = 30 if grid_key == 'ant' else 10

    # ── q map (with IHFC obs overplotted) ──────────────────────────────
    p_q, cb_q = save_maps_for_var(
        grid_obj, df_g, 'Q_mean', 'q (Similarity)',
        grid_key, 'sim', hf_cmap,
        hf_v_min * 1e-3, hf_v_max * 1e-3, hf_unit,
        ref_grid=ref, scatter_col='q', gridstep=gs,
    )

    # ── uncertainty (Q_std) map ─────────────────────────────────────────
    p_std, cb_std = save_maps_for_var(
        grid_obj, df_g, 'Q_std', 'std (Similarity)',
        grid_key, 'sim', std_cmap,
        std_v_min * 1e-3, std_v_max * 1e-3, std_unit,
        ref_grid=None, gridstep=gs,
    )

    # ── N_eff map ────────────────────────────────────────────────────────
    p_neff, cb_neff = save_maps_for_var(
        grid_obj, df_g, 'N_eff', 'N\_eff (Similarity)',
        grid_key, 'sim', 'cmc.batlow',
        1, 100, '',
        ref_grid=None, gridstep=gs,
    )

    sim_manifest.append(dict(
        method='sim', grid=grid_key, nc=str(nc_path),
        p_q=p_q, cb_q=cb_q,
        p_std=p_std, cb_std=cb_std,
        p_neff=p_neff, cb_neff=cb_neff,
    ))

print('\nSimilarity done.')

## 5 · QRF prediction

In [ ]:
from scipy.stats import skew as scipy_skew

QUANTILES_DENSE = np.linspace(0.02, 0.98, 49).tolist()
qrf_manifest = []

for grid_key, df_g_raw, grid_obj, epsg, parquet_path in TARGET_GRIDS:
    print(f'\n── QRF [{grid_key}] ──────────────────────────────────')
    df_g = df_g_raw.copy()

    missing_cols = [c for c in obs_model if c not in df_g.columns]
    for c in missing_cols:
        df_g[c] = np.nan
    if missing_cols:
        print(f'  WARNING: missing columns {missing_cols}')

    X_g = scaler.transform(df_g[obs_model].values.astype(np.float32))

    # ── quantile predictions ─────────────────────────────────────────────
    qrf_q = qrf.predict(X_g, quantiles=QUANTILES)     # (n, 5)
    print('  Sampling dense quantile grid...')
    leaf   = qrf.predict(X_g, quantiles=QUANTILES_DENSE)  # (n, 49)
    qrf_m  = leaf.mean(axis=1).astype(np.float32)
    qrf_s  = leaf.std(axis=1).astype(np.float32)
    q_skew = np.array([scipy_skew(row) for row in leaf], dtype=np.float32)
    q_hist_qrf = hist_from_samples(leaf)

    # ── distribution correction ──────────────────────────────────────────
    q50_raw  = qrf_q[:, 2].astype(np.float32)
    q50_corr = _apply_spline(q50_raw, qrf_correction_spline)
    q05_corr = _apply_spline(qrf_q[:, 0].astype(np.float32), qrf_correction_spline)
    q95_corr = _apply_spline(qrf_q[:, 4].astype(np.float32), qrf_correction_spline)

    # ── conformal PI ─────────────────────────────────────────────────────
    if qhat_qrf is not None:
        pi_lo = (q05_corr - qhat_qrf).astype(np.float32)
        pi_hi = (q95_corr + qhat_qrf).astype(np.float32)
    else:
        pi_lo = q05_corr
        pi_hi = q95_corr

    pi_width = (pi_hi - pi_lo).astype(np.float32)

    # ── NetCDF ───────────────────────────────────────────────────────────
    ds = make_ds_2d(df_g, {
        'Q_q05'     : q05_corr,
        'Q_q25'     : _apply_spline(qrf_q[:, 1].astype(np.float32), qrf_correction_spline),
        'Q_q50'     : q50_corr,
        'Q_q75'     : _apply_spline(qrf_q[:, 3].astype(np.float32), qrf_correction_spline),
        'Q_q95'     : q95_corr,
        'Q_mean'    : _apply_spline(qrf_m, qrf_correction_spline),
        'Q_std'     : qrf_s,
        'Q_skew'    : q_skew,
        'PI90_width': pi_width,
        'PI90_lo'   : pi_lo,
        'PI90_hi'   : pi_hi,
        'Q_hist'    : q_hist_qrf,
    }, attrs={
        'method'          : 'Quantile Random Forest (distribution-corrected)',
        'source'          : str(parquet_path),
        'quantiles'       : str(QUANTILES),
        'conformal_alpha' : CONFORMAL_ALPHA,
    }, crs_epsg=epsg)

    nc_path = out_dir / f'qrf_{grid_key}.nc'
    ds.to_netcdf(nc_path)
    print(f'  saved → {nc_path}')
    print(f'  Q_q50 range : {float(q50_corr.min())*1e3:.1f}–{float(q50_corr.max())*1e3:.1f} mW/m²')

    # ── inject for mapping ───────────────────────────────────────────────
    df_g['Q_q50']      = q50_corr
    df_g['Q_q05']      = q05_corr
    df_g['Q_q95']      = q95_corr
    df_g['PI90_width'] = pi_width
    df_g['Q_std']      = qrf_s

    gs = 30 if grid_key == 'ant' else 10

    p_q, cb_q = save_maps_for_var(
        grid_obj, df_g, 'Q_q50', 'q Q50 (QRF)',
        grid_key, 'qrf', hf_cmap,
        hf_v_min * 1e-3, hf_v_max * 1e-3, hf_unit,
        ref_grid=ref, scatter_col='q', gridstep=gs,
    )
    p_q05, cb_q05 = save_maps_for_var(
        grid_obj, df_g, 'Q_q05', 'q Q05 (QRF)',
        grid_key, 'qrf', hf_cmap,
        hf_v_min * 1e-3, hf_v_max * 1e-3, hf_unit,
        ref_grid=None, gridstep=gs,
    )
    p_q95, cb_q95 = save_maps_for_var(
        grid_obj, df_g, 'Q_q95', 'q Q95 (QRF)',
        grid_key, 'qrf', hf_cmap,
        hf_v_min * 1e-3, hf_v_max * 1e-3, hf_unit,
        ref_grid=None, gridstep=gs,
    )
    p_unc, cb_unc = save_maps_for_var(
        grid_obj, df_g, 'PI90_width', 'PI90 width (QRF)',
        grid_key, 'qrf', unc_cmap,
        unc_v_min * 1e-3, unc_v_max * 1e-3, unc_unit,
        ref_grid=None, gridstep=gs,
    )

    qrf_manifest.append(dict(
        method='qrf', grid=grid_key, nc=str(nc_path),
        p_q=p_q, cb_q=cb_q,
        p_q05=p_q05, cb_q05=cb_q05,
        p_q95=p_q95, cb_q95=cb_q95,
        p_unc=p_unc, cb_unc=cb_unc,
    ))

print('\nQRF done.')

## 6 · GBM prediction

In [ ]:
gbm_manifest = []

for grid_key, df_g_raw, grid_obj, epsg, parquet_path in TARGET_GRIDS:
    print(f'\n── GBM [{grid_key}] ──────────────────────────────────')
    df_g = df_g_raw.copy()

    missing_cols = [c for c in obs_model if c not in df_g.columns]
    for c in missing_cols:
        df_g[c] = np.nan
    if missing_cols:
        print(f'  WARNING: missing columns {missing_cols}')

    X_g = scaler.transform(df_g[obs_model].values.astype(np.float32))

    g_q05 = gbm_models[0.05].predict(X_g).astype(np.float32)
    g_q50 = gbm_models[0.50].predict(X_g).astype(np.float32)
    g_q95 = gbm_models[0.95].predict(X_g).astype(np.float32)

    # ── distribution correction ──────────────────────────────────────────
    g_q50_c = _apply_spline(g_q50, gbm_correction_spline)
    g_q05_c = _apply_spline(g_q05, gbm_correction_spline)
    g_q95_c = _apply_spline(g_q95, gbm_correction_spline)

    # ── conformal PI ─────────────────────────────────────────────────────
    if qhat_gbm is not None:
        pi_lo = (g_q05_c - qhat_gbm).astype(np.float32)
        pi_hi = (g_q95_c + qhat_gbm).astype(np.float32)
    else:
        pi_lo = g_q05_c
        pi_hi = g_q95_c

    pi_width = (pi_hi - pi_lo).astype(np.float32)

    # ── NetCDF ───────────────────────────────────────────────────────────
    ds = make_ds_2d(df_g, {
        'Q_q05'     : g_q05_c,
        'Q_q50'     : g_q50_c,
        'Q_q95'     : g_q95_c,
        'PI90_width': pi_width,
        'PI90_lo'   : pi_lo,
        'PI90_hi'   : pi_hi,
    }, attrs={
        'method'          : 'HistGradientBoosting (distribution-corrected + conformal)',
        'source'          : str(parquet_path),
        'conformal_alpha' : CONFORMAL_ALPHA,
    }, crs_epsg=epsg)

    nc_path = out_dir / f'gbm_{grid_key}.nc'
    ds.to_netcdf(nc_path)
    print(f'  saved → {nc_path}')
    print(f'  Q_q50 range : {float(g_q50_c.min())*1e3:.1f}–{float(g_q50_c.max())*1e3:.1f} mW/m²')

    # ── inject for mapping ───────────────────────────────────────────────
    df_g['Q_q50']      = g_q50_c
    df_g['Q_q05']      = g_q05_c
    df_g['Q_q95']      = g_q95_c
    df_g['PI90_width'] = pi_width

    gs = 30 if grid_key == 'ant' else 10

    p_q, cb_q = save_maps_for_var(
        grid_obj, df_g, 'Q_q50', 'q Q50 (GBM)',
        grid_key, 'gbm', hf_cmap,
        hf_v_min * 1e-3, hf_v_max * 1e-3, hf_unit,
        ref_grid=ref, scatter_col='q', gridstep=gs,
    )
    p_q05, cb_q05 = save_maps_for_var(
        grid_obj, df_g, 'Q_q05', 'q Q05 (GBM)',
        grid_key, 'gbm', hf_cmap,
        hf_v_min * 1e-3, hf_v_max * 1e-3, hf_unit,
        ref_grid=None, gridstep=gs,
    )
    p_q95, cb_q95 = save_maps_for_var(
        grid_obj, df_g, 'Q_q95', 'q Q95 (GBM)',
        grid_key, 'gbm', hf_cmap,
        hf_v_min * 1e-3, hf_v_max * 1e-3, hf_unit,
        ref_grid=None, gridstep=gs,
    )
    p_unc, cb_unc = save_maps_for_var(
        grid_obj, df_g, 'PI90_width', 'PI90 width (GBM)',
        grid_key, 'gbm', unc_cmap,
        unc_v_min * 1e-3, unc_v_max * 1e-3, unc_unit,
        ref_grid=None, gridstep=gs,
    )

    gbm_manifest.append(dict(
        method='gbm', grid=grid_key, nc=str(nc_path),
        p_q=p_q, cb_q=cb_q,
        p_q05=p_q05, cb_q05=cb_q05,
        p_q95=p_q95, cb_q95=cb_q95,
        p_unc=p_unc, cb_unc=cb_unc,
    ))

print('\nGBM done.')

## 7 · Output manifest

In [ ]:
import os

# ── NetCDF manifest ───────────────────────────────────────────────────────
nc_rows = []
for p in sorted(out_dir.glob('*.nc')):
    ds = xr.open_dataset(p)
    nc_rows.append({
        'file'    : p.name,
        'method'  : ds.attrs.get('method', ''),
        'variables': list(ds.data_vars),
        'size_MB' : round(os.path.getsize(p) / 1e6, 2),
    })
    ds.close()

nc_manifest = pd.DataFrame(nc_rows)
nc_manifest.to_csv(out_dir / 'manifest_nc.csv', index=False)
print('NetCDF manifest:')
print(nc_manifest.to_string(index=False))

# ── figure manifest ───────────────────────────────────────────────────────
fig_rows = []
for p in sorted(fig_dir.glob('*' + FIG_EXT)):
    fig_rows.append({'file': p.name, 'size_KB': round(os.path.getsize(p) / 1e3, 1)})

fig_manifest = pd.DataFrame(fig_rows)
fig_manifest.to_csv(fig_dir / 'manifest_figs.csv', index=False)
print(f'\nFigure manifest: {len(fig_manifest)} files → {fig_dir / "manifest_figs.csv"}')